# Assignment 6 — Medical Cost Analysis Modeling Notebook

This version keeps the previous seven-task project structure while adding the Week 10 modeling requirements: temporal-validation tuning, income and insurance variables, tuning improvement, two-scale evaluation, leakage check, saved fitted pipeline, and prediction function.

## Task 1 — Data Source, Reproducible Paths, and Codebook Verification

This task confirms that the notebook can run from the project folder instead of relying on a hard-coded Desktop path. It also verifies the added income and insurance variables against the HRS codebook.

In [ ]:
from pathlib import Path
import os
import sys

NOTEBOOK_LANGUAGE = 'en'
PREFERRED_VERSION_FOLDER = 'English Version'
OTHER_VERSION_FOLDER = 'Chinese Version'
PROJECT_DIR = None
start = Path.cwd().resolve()
home = Path.home().resolve()

candidate_dirs = []
for base in [start, *start.parents]:
    candidate_dirs.extend([
        base,
        base / PREFERRED_VERSION_FOLDER,
        base / 'homework4' / PREFERRED_VERSION_FOLDER,
        base / 'homework4_English' / PREFERRED_VERSION_FOLDER,
        base / 'homework4_Chinese' / PREFERRED_VERSION_FOLDER,
        base / 'Assignment 6 Modeling Notebook Package' / PREFERRED_VERSION_FOLDER,
        base / 'Assignment 6 Modeling Notebook Package' / 'Assignment 6 Modeling Notebook',
        base / 'Homework 3 Modeling Optimization',
    ])

candidate_dirs.extend([
    home / 'Documents' / 'data 975' / 'homework4' / PREFERRED_VERSION_FOLDER,
    home / 'Documents' / 'data 975' / 'Homework 3 Modeling Optimization',
    home / 'Downloads' / 'homework4' / PREFERRED_VERSION_FOLDER,
    home / 'Downloads' / 'homework4_English' / PREFERRED_VERSION_FOLDER,
    home / 'Downloads' / 'homework4_Chinese' / PREFERRED_VERSION_FOLDER,
    home / 'Downloads' / PREFERRED_VERSION_FOLDER,
])

search_roots = [start, home / 'Documents', home / 'Downloads']
for root in search_roots:
    if root.exists() and root.is_dir():
        try:
            for match in root.rglob(PREFERRED_VERSION_FOLDER):
                if len(match.relative_to(root).parts) <= 5:
                    candidate_dirs.append(match)
        except Exception:
            pass

seen = set()
for candidate in candidate_dirs:
    try:
        candidate = candidate.resolve()
    except Exception:
        continue
    if candidate in seen:
        continue
    seen.add(candidate)
    if candidate.name == OTHER_VERSION_FOLDER:
        continue
    if (candidate / 'scripts').exists() and (candidate / 'data' / 'homework 1').exists():
        PROJECT_DIR = candidate
        break

if PROJECT_DIR is None:
    checked = chr(10).join(str(p) for p in list(seen)[:25])
    raise FileNotFoundError(
        'Could not find the project folder containing scripts/ and data/homework 1/. '
        + 'Please open the notebook from inside homework4/' + PREFERRED_VERSION_FOLDER
        + ', or keep the notebook together with its scripts and data folders.'
        + chr(10) + chr(10) + 'First locations checked:' + chr(10) + checked
    )

OUTPUT_DIR = PROJECT_DIR / 'outputs'
FIG_DIR = OUTPUT_DIR / 'figures'
MODEL_DIR = OUTPUT_DIR / 'models'
os.environ.setdefault('MPLCONFIGDIR', str(OUTPUT_DIR / 'mpl_cache'))

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

script_path = str(PROJECT_DIR / 'scripts')
if script_path not in sys.path:
    sys.path.insert(0, script_path)

import modeling_optimization as opt

print('Current working directory:', start)
print('Project directory:', PROJECT_DIR)
print('Raw data folder exists:', (PROJECT_DIR / 'data' / 'homework 1').exists())
print('Scripts folder exists:', (PROJECT_DIR / 'scripts').exists())


**Task 1 Results**

**Project reproducibility check**

| item | status |
|:--|:--|
| Project folder detected | Yes |
| Relative data folder `data/homework 1` detected | Yes |
| Scripts folder detected | Yes |
| Desktop hard-coded data path required | No |

**Codebook verification for added variables**

| final_variable                 | source_variables      | source_section               | codebook_label                     | model_type         | cleaning_rule                                                                                       |
|:-------------------------------|:----------------------|:-----------------------------|:-----------------------------------|:-------------------|:----------------------------------------------------------------------------------------------------|
| medicare_coverage              | QN001 / RN001 / SN001 | HRS Core Interview Section N | MEDICARE COVERAGE                  | binary             | 1 is mapped to yes; 5 is mapped to no; negative and special missing values are set to missing.      |
| num_private_hi_plans           | QN023 / RN023 / SN023 | HRS Core Interview Section N | NUM PRIVATE HEALTH INS PLANS       | numeric count      | 0 and positive counts are kept; negative and 98/99-style special codes are set to missing.          |
| social_security_income_monthly | QQ085 / RQ085 / SQ085 | HRS Core Interview Section Q | R AMOUNT OF SS INCOME - LAST MONTH | continuous numeric | Valid dollar amounts are kept; negative and repeated 9 special missing values are set to missing.   |
| oop_rx_drugs_annualized        | QN180 / RN180 / SN180 | HRS Core Interview Section N | AMT PAY O-O-P RX DRUGS PER MONTH   | target component   | The monthly prescription drug amount is multiplied by 12 before being added to annual OOP spending. |

**Task 1 Interpretation**

The project now reads raw HRS files from the project folder instead of a fixed Desktop path. The added variables are verified by codebook label, source section, cleaning rule, and model type before entering the model.

## Task 2 — Data Cleaning and Target Variable Definition

This task cleans the full three-wave project data and defines total annual out-of-pocket medical spending. True zero-spending rows are retained. Missing spending components are recoded to zero only when the codebook gate variable clearly indicates no service use.

In [ ]:
raw, clean, optimized, cleaning_log = opt.build_optimized_dataset()

print(f'Raw selected rows: {len(raw):,}')
print(f'Cleaned modeling rows: {len(clean):,}')
print(f'Optimized modeling rows: {len(optimized):,}')

target_audit = pd.read_csv(OUTPUT_DIR / 'target_rule_audit.csv')
gate_audit = pd.read_csv(OUTPUT_DIR / 'spending_gate_zero_rule_audit.csv')

display(target_audit)
display(gate_audit)
display(cleaning_log)


**Task 2 Results**

**Target variable rule audit**

| target_rule_item                                                        |      n | interpretation                                                                                                                                    |
|:------------------------------------------------------------------------|-------:|:--------------------------------------------------------------------------------------------------------------------------------------------------|
| raw respondent-wave rows                                                |  48656 | All selected 2018, 2020, and 2022 respondent-wave rows before outcome filtering.                                                                  |
| rows with at least one usable spending component before gate-zero rules |  36437 | Rows with at least one observed spending amount before using service-use gate variables.                                                          |
| rows with no usable spending component before gate-zero rules           |  12219 | Rows that would be dropped under the earlier conservative rule.                                                                                   |
| spending components set to 0 using codebook gate variables              | 244637 | Component-level missing values recoded to 0 when the codebook gate variable clearly indicates no service/use.                                     |
| rows with no usable spending component after gate-zero rules            |    388 | Rows still lacking any usable spending component after codebook gate variables are applied.                                                       |
| true zero-spending rows retained after gate-zero rules                  |  13120 | Rows with usable spending-component information summing to zero after gate-zero recoding; these remain in the dataset and are modeled with log1p. |
| clean modeling rows after outcome rule                                  |  48268 | Rows available after dropping only remaining missing outcomes, not zero-dollar outcomes.                                                          |

**Gate-zero component audit**

| spending_component       | gate_variable                  |   component_missing_set_to_zero_n |   component_missing_set_to_zero_rate |
|:-------------------------|:-------------------------------|----------------------------------:|-------------------------------------:|
| oop_hospital             | gate_hospital_overnight        |                             37104 |                               0.7626 |
| oop_nursing_home         | gate_nursing_home_overnight    |                             46668 |                               0.9591 |
| oop_outpatient_surgery   | gate_outpatient_surgery        |                             39466 |                               0.8111 |
| oop_doctor_visits        | gate_doctor_visit_count        |                              5444 |                               0.1119 |
| oop_dental               | gate_dental_visit              |                             17497 |                               0.3596 |
| oop_rx_drugs_monthly     | gate_takes_rx_regularly        |                              8617 |                               0.1771 |
| oop_home_health          | gate_home_health_service       |                             43690 |                               0.8979 |
| oop_other_health_service | gate_paid_other_health_service |                              5418 |                               0.1114 |
| oop_other_medical        | gate_other_medical_expense     |                             40733 |                               0.8372 |

**Cleaning log**

| step                       |   rows |   columns | note                                                                                                                                                                                                                                                                                                                                             |
|:---------------------------|-------:|----------:|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| raw_combined               |  48656 |        43 | Combined 2018, 2020, and 2022 selected raw variables.                                                                                                                                                                                                                                                                                            |
| clean_spending             |  48656 |        46 | Cleaned spending components with variable-specific missing codes and used codebook gate variables to set clear no-service components to 0. HRS codebook labels QN180/RN180/SN180 as AMT PAY O-O-P RX DRUGS PER MONTH; HRS notes that values are converted to monthly amounts where possible, so this project annualizes it by multiplying by 12. |
| binary_health_variables    |  48656 |        58 | Converted chronic disease indicators with revised yes/no codes and filled smoking variables within person across waves.                                                                                                                                                                                                                          |
| bmi_engineering            |  48656 |        62 | Calculated BMI and obesity indicator from self-reported height and weight.                                                                                                                                                                                                                                                                       |
| demographic_and_log_target |  48656 |        66 | Decoded demographics as categorical variables and created log1p target so zero-spending respondents stay in the dataset.                                                                                                                                                                                                                         |
| drop_missing_outcome_only  |  48268 |        66 | Removed 388 rows without usable spending components; zero-spending and low-spending respondents are retained.                                                                                                                                                                                                                                    |

**Task 2 Interpretation**

The final modeling dataset contains **48,268** rows. The codebook-based gate-zero rule retains true zero-spending respondents and drops only the remaining records without usable spending outcome information.

## Task 3 — EDA Analysis

This task keeps the Homework 2 EDA logic and uses visual evidence to show the relationship between medical spending, health status, chronic disease burden, income, and insurance variables. The interpretation is correlational and predictive, not causal.

In [ ]:
added_missingness, income_insurance_eda = opt.make_income_insurance_eda(optimized)

display(added_missingness.round(4))
display(income_insurance_eda.round(2))

# Spending distribution before and after log transformation.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(optimized['total_oop_medical_spending'], bins=70, color='#315C8C', edgecolor='white')
axes[0].set_title(f"Raw total OOP spending, skew={optimized['total_oop_medical_spending'].skew():.2f}")
axes[0].set_xlabel('Total out-of-pocket spending')
axes[0].set_ylabel('Count')
axes[1].hist(optimized['log_total_oop_medical_spending'], bins=70, color='#5F8F4E', edgecolor='white')
axes[1].set_title(f"log1p spending, skew={optimized['log_total_oop_medical_spending'].skew():.2f}")
axes[1].set_xlabel('log1p(total OOP spending)')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()

# Scatterplots: continuous predictors and log medical spending.
scatter_specs = [
    ('age', 'Age'),
    ('bmi_self_reported', 'BMI'),
    ('cigarettes_per_day', 'Cigarettes per day'),
    ('chronic_condition_count', 'Chronic condition count'),
    ('num_private_hi_plans', 'Number of private insurance plans'),
    ('social_security_income_monthly', 'Monthly Social Security income'),
]
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
for ax, (col, label) in zip(axes.ravel(), scatter_specs):
    temp = optimized[[col, 'log_total_oop_medical_spending', 'wave']].dropna()
    if len(temp) > 7000:
        temp = temp.sample(7000, random_state=42)
    colors = temp['wave'].map({2018: '#4E79A7', 2020: '#F28E2B', 2022: '#59A14F'})
    ax.scatter(temp[col], temp['log_total_oop_medical_spending'], s=8, alpha=0.28, c=colors)
    ax.set_title(f'{label} vs log spending')
    ax.set_xlabel(label)
    ax.set_ylabel('log1p spending')
plt.tight_layout()
plt.show()

# Chronic-condition burden: table plus bar and line charts.
chronic = optimized.groupby('chronic_condition_count')['total_oop_medical_spending'].agg(
    n='size', median_spending='median', mean_spending='mean'
).reset_index().sort_values('chronic_condition_count')
display(chronic.round(2))

fig, ax = plt.subplots(figsize=(11, 5.5))
bars = ax.bar(chronic['chronic_condition_count'].astype(int).astype(str), chronic['median_spending'], color='#6B8FB3')
ax.plot(chronic['chronic_condition_count'].astype(int).astype(str), chronic['median_spending'], color='#C0392B', marker='o', linewidth=2)
ax.set_title('Median spending increases as chronic conditions accumulate')
ax.set_xlabel('Number of chronic conditions')
ax.set_ylabel('Median total OOP spending')
ax.bar_label(bars, fmt='$%.0f', padding=3, fontsize=9)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(chronic['chronic_condition_count'], chronic['median_spending'], marker='o', color='#315C8C', linewidth=2)
ax.set_title('Median spending rises with chronic-condition burden')
ax.set_xlabel('Number of chronic conditions')
ax.set_ylabel('Median total OOP spending')
plt.tight_layout()
plt.show()

# Binary health and insurance group comparisons.
group_specs = [
    ('obese', 'Obese'),
    ('current_smoker', 'Current smoker'),
    ('medicare_coverage', 'Medicare coverage'),
]
group_rows = []
for col, label in group_specs:
    summary = optimized.groupby(col, dropna=False)['total_oop_medical_spending'].agg(
        n='size', median_spending='median', mean_spending='mean'
    ).reset_index().rename(columns={col: 'group_value'})
    summary['factor'] = label
    group_rows.append(summary)
group_summary = pd.concat(group_rows, ignore_index=True)
display(group_summary.round(2))

plot_group = group_summary[group_summary['group_value'].isin([0.0, 1.0])].copy()
plot_group['group_label'] = plot_group['group_value'].map({0.0: 'No', 1.0: 'Yes'})
pivot = plot_group.pivot(index='factor', columns='group_label', values='median_spending')
fig, ax = plt.subplots(figsize=(9, 5))
pivot.plot(kind='bar', ax=ax, color=['#6B8FB3', '#C95F4F'])
ax.set_title('Median OOP spending by health and insurance groups')
ax.set_xlabel('Risk or coverage factor')
ax.set_ylabel('Median total OOP spending')
ax.tick_params(axis='x', rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt='$%.0f', padding=3, fontsize=8)
plt.tight_layout()
plt.show()

# Private insurance plan count.
private_summary = optimized.copy()
private_summary['private_plan_group'] = private_summary['num_private_hi_plans'].clip(upper=3).map({0: '0', 1: '1', 2: '2', 3: '3+'})
private_summary = private_summary.groupby('private_plan_group', dropna=False)['total_oop_medical_spending'].agg(
    n='size', median_spending='median', mean_spending='mean'
).reset_index()
display(private_summary.round(2))

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(private_summary['private_plan_group'].astype(str), private_summary['median_spending'], color='#76B7B2')
ax.set_title('Median spending by number of private insurance plans')
ax.set_xlabel('Number of private health insurance plans')
ax.set_ylabel('Median total OOP spending')
ax.bar_label(bars, fmt='$%.0f', padding=3, fontsize=9)
plt.tight_layout()
plt.show()

# Social Security income quartiles.
income_plot = optimized.copy()
income_plot['ss_income_quartile'] = pd.qcut(income_plot['social_security_income_monthly'], q=4, duplicates='drop')
income_summary = income_plot.groupby('ss_income_quartile', dropna=False)['total_oop_medical_spending'].agg(
    n='size', median_spending='median', mean_spending='mean'
).reset_index()
display(income_summary.round(2))

fig, ax = plt.subplots(figsize=(10, 5))
valid_income = income_summary[~income_summary['ss_income_quartile'].astype(str).eq('nan')].copy()
bars = ax.bar(range(len(valid_income)), valid_income['median_spending'], color='#9C755F')
ax.set_title('Median spending by Social Security income quartile')
ax.set_xlabel('Monthly Social Security income group')
ax.set_ylabel('Median total OOP spending')
ax.set_xticks(range(len(valid_income)))
ax.set_xticklabels([str(x).replace(', ', ',\n') for x in valid_income['ss_income_quartile']], fontsize=8)
ax.bar_label(bars, fmt='$%.0f', padding=3, fontsize=8)
plt.tight_layout()
plt.show()

# Yearly spending and zero-spending rate.
wave_summary = optimized.groupby('wave')['total_oop_medical_spending'].agg(
    n='size', median_spending='median', mean_spending='mean', zero_rate=lambda s: (s == 0).mean()
).reset_index()
display(wave_summary.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
bars = axes[0].bar(wave_summary['wave'].astype(str), wave_summary['median_spending'], color='#4E79A7')
axes[0].set_title('Median OOP spending by wave')
axes[0].set_xlabel('Wave')
axes[0].set_ylabel('Median total OOP spending')
axes[0].bar_label(bars, fmt='$%.0f', padding=3)
axes[1].plot(wave_summary['wave'].astype(str), wave_summary['zero_rate'] * 100, marker='o', color='#E15759', linewidth=2)
axes[1].set_title('Zero-spending rate by wave')
axes[1].set_xlabel('Wave')
axes[1].set_ylabel('Zero-spending rate (%)')
for x, y in zip(wave_summary['wave'].astype(str), wave_summary['zero_rate'] * 100):
    axes[1].annotate(f'{y:.1f}%', (x, y), textcoords='offset points', xytext=(0, 7), ha='center')
plt.tight_layout()
plt.show()

# Correlation heatmap for numeric predictors and spending.
corr_cols = [
    'log_total_oop_medical_spending',
    'total_oop_medical_spending',
    'age',
    'bmi_self_reported',
    'cigarettes_per_day',
    'chronic_condition_count',
    'num_private_hi_plans',
    'social_security_income_monthly',
    'medicare_coverage',
    'obese',
    'current_smoker',
]
corr = optimized[corr_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10.5, 8.5))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_title('Correlation between medical spending and selected predictors')
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(corr.index, fontsize=8)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

**Task 3 Results**

**Added variable missingness**

| feature                        |     n |   nonmissing_n |   missing_n |   missing_rate |
|:-------------------------------|------:|---------------:|------------:|---------------:|
| medicare_coverage              | 48268 |          47987 |         281 |         0.0058 |
| num_private_hi_plans           | 48268 |          47567 |         701 |         0.0145 |
| social_security_income_monthly | 48268 |          25394 |       22874 |         0.4739 |

**Income and insurance EDA summary**

| group             |     n |   median_spending |   mean_spending | variable                       |
|:------------------|------:|------------------:|----------------:|:-------------------------------|
| No                | 19893 |            500    |         2018.6  | medicare_coverage              |
| Yes               | 28094 |            500    |         1963.83 | medicare_coverage              |
| nan               |   281 |              0    |          714.1  | medicare_coverage              |
| 0                 | 25418 |            300    |         1587.39 | num_private_hi_plans           |
| 1                 | 20586 |            780    |         2462.8  | num_private_hi_plans           |
| 2                 |  1183 |            750    |         2436.46 | num_private_hi_plans           |
| 3+                |   380 |           1040    |         2137.15 | num_private_hi_plans           |
| nan               |   701 |              0    |         1122.1  | num_private_hi_plans           |
| (-0.001, 865.0]   |  6361 |            200    |         1462.06 | social_security_income_monthly |
| (865.0, 1290.0]   |  6337 |            400    |         1733.14 | social_security_income_monthly |
| (1290.0, 1700.0]  |  6559 |            736    |         2195.63 | social_security_income_monthly |
| (1700.0, 38000.0] |  6137 |            960    |         2707.1  | social_security_income_monthly |
| nan               | 22874 |            448.98 |         1933.67 | social_security_income_monthly |

**Task 3 Interpretation**

The EDA combines tables and figures. The missingness table supports whether the added variables are usable, while the spending summaries and charts show how insurance, income, chronic disease burden, and health status relate to out-of-pocket spending.

## Task 4 — Feature Engineering and Variable Type Processing

This task assigns features to the correct preprocessing group. Numeric, binary, and categorical variables are handled separately, and categorical variables are one-hot encoded. This is direct evidence for SLO 2.

In [ ]:
base_features, base_numeric, base_binary, base_categorical, base_schema = opt.make_feature_schema(include_added_variables=False)
plus_features, plus_numeric, plus_binary, plus_categorical, plus_schema = opt.make_feature_schema(include_added_variables=True)

print('Base feature count:', len(base_features))
print('Expanded feature count:', len(plus_features))

display(base_schema)
display(plus_schema)

feature_missingness = optimized[plus_features].isna().mean().reset_index()
feature_missingness.columns = ['feature', 'missing_rate']
display(feature_missingness.round(4))


**Task 4 Results**

**Base variable type schema**

| feature                 | model_type   |
|:------------------------|:-------------|
| age                     | numeric      |
| bmi_self_reported       | numeric      |
| cigarettes_per_day      | numeric      |
| chronic_condition_count | numeric      |
| obese                   | binary       |
| current_smoker          | binary       |
| sex                     | categorical  |
| race                    | categorical  |
| education               | categorical  |

**Expanded variable type schema**

| feature                        | model_type   |
|:-------------------------------|:-------------|
| age                            | numeric      |
| bmi_self_reported              | numeric      |
| cigarettes_per_day             | numeric      |
| chronic_condition_count        | numeric      |
| num_private_hi_plans           | numeric      |
| social_security_income_monthly | numeric      |
| obese                          | binary       |
| current_smoker                 | binary       |
| medicare_coverage              | binary       |
| sex                            | categorical  |
| race                           | categorical  |
| education                      | categorical  |

**Task 4 Interpretation**

This table is the main SLO 2 evidence. Numeric, binary, and categorical variables are processed separately. Race and education are treated as categorical variables, Medicare coverage is binary, and income/private-plan count are numeric.

## Task 5 — Model Selection and Model Combination

This task selects the model using temporal CV, not the 2022 test set. It compares the base feature set with the expanded feature set that includes income and insurance variables.

In [ ]:
cv_all, test_all, ctx_base, ctx_plus = opt.compare_base_vs_plus(optimized)

display(cv_all.round(4))

cv_chosen = cv_all.query("feature_set == 'plus_income_insurance'").sort_values('temporal_cv_rmse_log_mean').iloc[0]['model']
print('Model chosen by temporal CV among expanded-feature models:', cv_chosen)

display(Image(filename=str(FIG_DIR / 'optimized_temporal_cv_model_comparison.png')))


**Task 5 Results**

| model             | feature_set           |   temporal_cv_rmse_log_mean |   temporal_cv_mae_log_mean |
|:------------------|:----------------------|----------------------------:|---------------------------:|
| Gradient boosting | plus_income_insurance |                      3.0427 |                     2.4763 |
| Random forest     | plus_income_insurance |                      3.0517 |                     2.4825 |
| Ridge regression  | plus_income_insurance |                      3.0912 |                     2.5509 |
| Elastic Net       | plus_income_insurance |                      3.0916 |                     2.5554 |
| Gradient boosting | base                  |                      3.1168 |                     2.559  |
| Random forest     | base                  |                      3.1269 |                     2.5606 |
| Ridge regression  | base                  |                      3.1287 |                     2.5793 |
| Elastic Net       | base                  |                      3.1296 |                     2.5852 |
| Baseline mean     | base                  |                      3.2971 |                     2.7853 |
| Baseline mean     | plus_income_insurance |                      3.2971 |                     2.7853 |
| Two-part hurdle   | plus_income_insurance |                      3.4703 |                     2.5298 |
| Two-part hurdle   | base                  |                      3.5025 |                     2.5512 |
| Tweedie regressor | plus_income_insurance |                      4.006  |                     2.8708 |
| Tweedie regressor | base                  |                      4.0358 |                     2.895  |

**Task 5 Interpretation**

This table is the main evidence for model selection. The final model is selected by `temporal_cv_rmse_log_mean`, not by the 2022 test set. Among the expanded-feature models, **Gradient boosting** performs best with CV RMSE log = **3.0427**, which is lower than Random Forest at **3.0517** and lower than the best base-feature result at **3.1168**. This supports adding income and insurance variables to the model.

## Task 6 — Model Evaluation, Extreme-Value Check, and Tuning

This task uses 2022 only as the final held-out test set. It reports both dollar-scale and log-scale metrics, checks whether extreme bills dominate the results, and tunes the CV-selected model with the same temporal CV design.

In [ ]:
robustness = opt.outlier_robustness_check(ctx_plus, cv_chosen)
tuned_results, best_model, model_path = opt.tune_cv_chosen_model(optimized, cv_chosen)

display(test_all[['model', 'feature_set', 'MAE_dollars', 'RMSE_dollars', 'R2_dollars', 'RMSE_log', 'R2_log']].round(4))
display(robustness.round(4))
display(tuned_results.round(6))

display(Image(filename=str(FIG_DIR / 'optimized_test_mae_model_comparison.png')))
print('Saved tuned model:', model_path)


In [ ]:
# Visual comparison of log-scale and dollar-scale model performance.
test_plot = test_all.copy()
test_plot['label'] = test_plot['model'] + '\n' + test_plot['feature_set'].str.replace('_', ' ')
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
bars0 = axes[0].bar(test_plot['label'], test_plot['RMSE_log'], color='#4E79A7')
axes[0].set_title('2022 test RMSE on log target')
axes[0].set_ylabel('RMSE log')
axes[0].tick_params(axis='x', rotation=45, labelsize=8)
axes[0].bar_label(bars0, fmt='%.2f', padding=2, fontsize=7)
bars1 = axes[1].bar(test_plot['label'], test_plot['R2_dollars'], color='#E15759')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('2022 test R2 on dollar scale')
axes[1].set_ylabel('R2 dollars')
axes[1].tick_params(axis='x', rotation=45, labelsize=8)
axes[1].bar_label(bars1, fmt='%.2f', padding=2, fontsize=7)
plt.tight_layout()
plt.show()

# Outlier robustness visualization.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
bars0 = axes[0].bar(robustness['rule'], robustness['RMSE_dollars'], color=['#4E79A7', '#F28E2B'])
axes[0].set_title('RMSE before and after trimming top 1%')
axes[0].set_ylabel('RMSE dollars')
axes[0].tick_params(axis='x', rotation=15, labelsize=8)
axes[0].bar_label(bars0, fmt='$%.0f', padding=3)
bars1 = axes[1].bar(robustness['rule'], robustness['MAE_dollars'], color=['#59A14F', '#9C755F'])
axes[1].set_title('MAE before and after trimming top 1%')
axes[1].set_ylabel('MAE dollars')
axes[1].tick_params(axis='x', rotation=15, labelsize=8)
axes[1].bar_label(bars1, fmt='$%.0f', padding=3)
plt.tight_layout()
plt.show()

# Tuning improvement visualization.
tune_plot = pd.DataFrame({
    'model_version': ['Default Gradient Boosting', 'Tuned Gradient Boosting'],
    'temporal_cv_rmse_log': [
        tuned_results.loc[0, 'temporal_cv_default_rmse_log'],
        tuned_results.loc[0, 'temporal_cv_best_rmse_log'],
    ]
})
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(tune_plot['model_version'], tune_plot['temporal_cv_rmse_log'], color=['#BAB0AC', '#4E79A7'])
ax.set_title('Temporal CV RMSE before and after tuning')
ax.set_ylabel('Temporal CV RMSE log')
ax.bar_label(bars, fmt='%.4f', padding=3)
plt.tight_layout()
plt.show()

# Additional model-metric visualizations that were previously only shown in tables.
# 1) 2022 test RMSE in dollars by model.
test_plot = test_all.copy()
test_plot['label'] = test_plot['model'] + '\n' + test_plot['feature_set'].str.replace('_', ' ')
fig, ax = plt.subplots(figsize=(13, 5.5))
bars = ax.bar(test_plot['label'], test_plot['RMSE_dollars'], color='#B07AA1')
ax.set_title('2022 Test RMSE in Dollars by Model')
ax.set_ylabel('RMSE in dollars')
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.bar_label(bars, fmt='$%.0f', padding=2, fontsize=7)
plt.tight_layout()
plt.show()

# 2) R2 comparison on log scale and dollar scale.
r2_plot = test_all[['model', 'feature_set', 'R2_log', 'R2_dollars']].copy()
r2_plot['label'] = r2_plot['model'] + '\n' + r2_plot['feature_set'].str.replace('_', ' ')
fig, ax = plt.subplots(figsize=(13, 5.5))
x = range(len(r2_plot))
width = 0.38
bars1 = ax.bar([i - width/2 for i in x], r2_plot['R2_log'], width=width, label='R2 log target', color='#4E79A7')
bars2 = ax.bar([i + width/2 for i in x], r2_plot['R2_dollars'], width=width, label='R2 dollars', color='#E15759')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('R2 Comparison: Log Target vs Dollar Scale')
ax.set_ylabel('R2')
ax.set_xticks(list(x))
ax.set_xticklabels(r2_plot['label'], rotation=45, ha='right', fontsize=8)
ax.legend()
ax.bar_label(bars1, fmt='%.2f', padding=2, fontsize=7)
ax.bar_label(bars2, fmt='%.2f', padding=2, fontsize=7)
plt.tight_layout()
plt.show()

# 3) Temporal CV MAE on log target by model.
cv_plot = cv_all.copy()
cv_plot['label'] = cv_plot['model'] + '\n' + cv_plot['feature_set'].str.replace('_', ' ')
fig, ax = plt.subplots(figsize=(13, 5.5))
bars = ax.bar(cv_plot['label'], cv_plot['temporal_cv_mae_log_mean'], color='#F28E2B')
ax.set_title('Temporal CV MAE on Log Target by Model')
ax.set_ylabel('MAE on log1p spending')
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=7)
plt.tight_layout()
plt.show()

# 4) Base feature set vs expanded income/insurance feature set improvement by model.
paired = cv_all.pivot(index='model', columns='feature_set', values='temporal_cv_rmse_log_mean').dropna()
paired['rmse_log_improvement_from_added_variables'] = paired['base'] - paired['plus_income_insurance']
paired = paired.sort_values('rmse_log_improvement_from_added_variables', ascending=False).reset_index()
display(paired.round(4))

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(paired['model'], paired['rmse_log_improvement_from_added_variables'], color='#59A14F')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('CV RMSE Log Improvement After Adding Income and Insurance Variables')
ax.set_ylabel('Base RMSE log - Expanded RMSE log')
ax.tick_params(axis='x', rotation=30, labelsize=9)
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)
plt.tight_layout()
plt.show()

**Task 6 Results: 2022 Test Set**

| model             | feature_set           |   MAE_dollars |   RMSE_dollars |   R2_dollars |   RMSE_log |   R2_log |
|:------------------|:----------------------|--------------:|---------------:|-------------:|-----------:|---------:|
| Gradient boosting | plus_income_insurance |       1811.65 |        7124.72 |      -0.0479 |     3.1494 |   0.149  |
| Random forest     | plus_income_insurance |       1811.71 |        7120.06 |      -0.0465 |     3.1561 |   0.1454 |
| Ridge regression  | plus_income_insurance |       1873.8  |        7510.76 |      -0.1645 |     3.1953 |   0.124  |
| Elastic Net       | plus_income_insurance |       1882.5  |        7656.36 |      -0.2101 |     3.1973 |   0.1229 |
| Random forest     | base                  |       1833.33 |        7142.9  |      -0.0532 |     3.2353 |   0.1019 |
| Gradient boosting | base                  |       1833.37 |        7142.7  |      -0.0532 |     3.2358 |   0.1016 |
| Ridge regression  | base                  |       1836.34 |        7141.14 |      -0.0527 |     3.2399 |   0.0994 |
| Elastic Net       | base                  |       1840.15 |        7146.31 |      -0.0542 |     3.2429 |   0.0977 |
| Baseline mean     | base                  |       1892.43 |        7184.03 |      -0.0654 |     3.4285 |  -0.0086 |
| Baseline mean     | plus_income_insurance |       1892.43 |        7184.03 |      -0.0654 |     3.4285 |  -0.0086 |
| Two-part hurdle   | plus_income_insurance |       1825.46 |        7025.24 |      -0.0188 |     3.6812 |  -0.1627 |
| Two-part hurdle   | base                  |       1832.94 |        7035.09 |      -0.0217 |     3.7082 |  -0.1798 |
| Tweedie regressor | plus_income_insurance |       2292.85 |        6913.74 |       0.0133 |     4.2305 |  -0.5355 |
| Tweedie regressor | base                  |       2310.3  |        6918.94 |       0.0118 |     4.2621 |  -0.5586 |

**Task 6 Results: Extreme-Value Robustness Check**

| model                          |   MAE_dollars |   RMSE_dollars |   R2_dollars |   RMSE_log |   R2_log |     n | rule                                                          |   top_1_percent_threshold |
|:-------------------------------|--------------:|---------------:|-------------:|-----------:|---------:|------:|:--------------------------------------------------------------|--------------------------:|
| Gradient boosting full 2022    |       1811.65 |        7124.72 |      -0.0479 |     3.1494 |    0.149 | 15633 | Full 2022 test set                                            |                       nan |
| Gradient boosting trimmed 2022 |       1334.54 |        2745.86 |      -0.1711 |     3.1247 |    0.145 | 15467 | Drops top 1 percent of 2022 OOP spending for robustness check |                     20000 |

**Task 6 Results: Tuned Final Model**

| model                   |   MAE_dollars |   RMSE_dollars |   R2_dollars |   RMSE_log |   R2_log | best_params                                                                         |   temporal_cv_default_rmse_log |   temporal_cv_default_mae_log |   temporal_cv_best_rmse_log |   temporal_cv_rmse_log_improvement |   temporal_cv_rmse_log_improvement_percent |
|:------------------------|--------------:|---------------:|-------------:|-----------:|---------:|:------------------------------------------------------------------------------------|-------------------------------:|------------------------------:|----------------------------:|-----------------------------------:|-------------------------------------------:|
| Tuned Gradient boosting |       1812.18 |        7125.65 |    -0.048154 |     3.1489 | 0.149243 | {'model__learning_rate': 0.03, 'model__max_iter': 200, 'model__max_leaf_nodes': 31} |                        3.04296 |                       2.47376 |                     3.04131 |                           0.001655 |                                   0.054379 |

**Task 6 Interpretation**

The 2022 test set is used only for final reporting, not for model selection. Gradient boosting has the strongest test performance on the log target, with RMSE log = **3.1494** and R2 log = **0.1490**. However, dollar-scale R2 is negative, which shows that raw dollar spending is strongly affected by extreme high-cost cases. After removing the top 1% of 2022 medical spending, RMSE falls from about **$7,124.72** to about **$2,745.86**, confirming that dollar-scale RMSE is highly sensitive to outliers. Tuning reduces CV RMSE log from **3.042962** to **3.041307**, a very small improvement, so the default Gradient Boosting model was already close to optimal for this feature set.

## Task 7 — Saved Pipeline, Prediction Function, and Leakage Explanation

This final task saves the fitted pipeline and provides a prediction function. The pipeline includes imputation, scaling, one-hot encoding, and the final model, so training and prediction use the same preprocessing rules. The current model does not use `HHID`, `PN`, or previous-wave spending as predictors, so the temporal split is acceptable. If lagged spending is added later, a person-grouped temporal split should be used.

In [ ]:
features, *_ = opt.make_feature_schema(include_added_variables=True)
example_prediction = opt.make_deployment_example(best_model, features)
display(example_prediction.round(3))

new_person = example_prediction.drop(columns=['predicted_oop_dollars']).iloc[0].to_dict()
predicted_from_saved_pipeline = opt.predict_oop_dollars_from_raw(new_person, MODEL_DIR / 'hrs_oop_spending_final_optimized.joblib')
print(f'Prediction from saved pipeline: ${predicted_from_saved_pipeline:,.2f}')


**Task 7 Results**

**Deployment prediction example**

|   age |   bmi_self_reported |   cigarettes_per_day |   chronic_condition_count |   num_private_hi_plans |   social_security_income_monthly |   obese |   current_smoker |   medicare_coverage | sex    | race   | education   |   predicted_oop_dollars |
|------:|--------------------:|---------------------:|--------------------------:|-----------------------:|---------------------------------:|--------:|-----------------:|--------------------:|:-------|:-------|:------------|------------------------:|
|    68 |                31.2 |                    0 |                         3 |                      1 |                             1600 |       1 |                0 |                   1 | female | white  | degree_3    |                 744.952 |

**Saved fitted pipeline**

| item | value |
|:--|:--|
| Saved model file | `outputs/models/hrs_oop_spending_final_optimized.joblib` |
| Prediction function | `predict_oop_dollars_from_raw()` |
| Train/serve consistency | Same fitted pipeline handles imputation, scaling, one-hot encoding, and prediction |
| Current leakage check | No `HHID`, `PN`, `HHIDPN`, or previous-wave spending used as predictors |

**Task 7 Interpretation**

The saved pipeline allows new raw feature values to be processed with the same rules used during model training. This supports reproducibility and deployment consistency.